# Train YOLO on Dataset-3 (Google Colab with GPU)

### Quick Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run each cell in order.
3. Upload Dataset-3_yolo.zip (45 MB) when prompted in Cell 3.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Install Ultralytics YOLO
!pip install -q ultralytics

import torch
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Extract Dataset-3_yolo.zip and configure data.yaml
import os
from google.colab import files

if not os.path.exists("Dataset-3_yolo.zip"):
    print("Please upload Dataset-3_yolo.zip from your computer...")
    uploaded = files.upload()

# Linux native unzip
!unzip -q -o Dataset-3_yolo.zip -d .

# Ensure exact Colab path in data.yaml
yaml_content = '''path: /content/Dataset-3_yolo
train: images/train
val: images/val
test: images/test

nc: 1
names: ['pothole']
'''

with open('/content/Dataset-3_yolo/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("Extraction & data.yaml configuration complete:")
!cat /content/Dataset-3_yolo/data.yaml


In [ ]:
# Cell 4: Train YOLOv8s
from ultralytics import YOLO

# Load pretrained YOLOv8s weights
model = YOLO("yolov8s.pt")

# Train for 50 epochs (takes ~5-7 minutes on Colab T4 GPU)
results = model.train(
    data="/content/Dataset-3_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="runs/colab_dataset3",
    name="exp_yolov8s",
    seed=42,
    plots=True
)

In [ ]:
# Cell 5: Evaluate on Validation and Test Splits
best_weights = f"{model.trainer.save_dir}/weights/best.pt"
best_model = YOLO(best_weights)

print("=" * 50)
print("VALIDATION SET EVALUATION:")
print("=" * 50)
val_res = best_model.val(data="/content/Dataset-3_yolo/data.yaml", split="val")
print(f"Precision:   {val_res.box.p.mean():.4f}")
print(f"Recall:      {val_res.box.r.mean():.4f}")
print(f"mAP@0.5:     {val_res.box.map50:.4f}")
print(f"mAP@0.5:0.95:{val_res.box.map:.4f}")

print("\n" + "=" * 50)
print("TEST SET EVALUATION:")
print("=" * 50)
test_res = best_model.val(data="/content/Dataset-3_yolo/data.yaml", split="test")
print(f"Precision:   {test_res.box.p.mean():.4f}")
print(f"Recall:      {test_res.box.r.mean():.4f}")
print(f"mAP@0.5:     {test_res.box.map50:.4f}")
print(f"mAP@0.5:0.95:{test_res.box.map:.4f}")

In [ ]:
# ==============================================================================
# SECTION B: TRANSFORM DATASET-3 (DAY -> SYNTHETIC NIGHT)
# ==============================================================================
import os, cv2, math, shutil
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

W_BGR = np.array([0.114, 0.587, 0.299], np.float32)
GAMMA = 2.2
HEAD_TINT = np.array([0.80, 0.93, 1.00], np.float32)
AMB_TINT = np.array([1.20, 1.00, 0.85], np.float32)

PRESETS = {
    "dusk":  dict(target_mean=70, peak=1500, read_std=0.004, beam=1.2, ambient=0.25),
    "night": dict(target_mean=40, peak=800,  read_std=0.006, beam=1.8, ambient=0.12),
}

def mean_luma(img):
    return float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())

def _exposure_for_target(lin, target):
    small = cv2.resize(lin, (96, 96), interpolation=cv2.INTER_AREA)
    lo, hi = -14.0, 14.0
    for _ in range(32):
        mid = (lo + hi) / 2
        enc = np.clip(small * (2.0 ** mid), 0, 1) ** (1 / GAMMA)
        if 255.0 * float((enc @ W_BGR).mean()) < target:
            lo = mid
        else:
            hi = mid
    return 2.0 ** ((lo + hi) / 2)

def to_night(img, rng, target_mean=40, peak=800, read_std=0.006, beam=1.8, ambient=0.12,
             cx=0.5, cy=1.0, illum_strength=0.85, illum_sigma_frac=0.08, desat=0.3, max_ratio=0.7):
    h, w = img.shape[:2]
    before = mean_luma(img)
    target = float(min(target_mean, max_ratio * before))
    x = (img.astype(np.float32) / 255.0) ** GAMMA

    # Daylight removal
    lum = x @ W_BGR
    illum = cv2.GaussianBlur(lum, (0, 0), sigmaX=max(h, w) * illum_sigma_frac)
    base = max(float(illum.mean()), 1e-4)
    shading = np.clip(illum / base, 0.25, 4.0) ** illum_strength
    reflect = np.clip(x / (shading[..., None] * base), 0, 4.0)

    # Headlight + ambient
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    d = ((xx / max(w - 1, 1) - cx) / 0.5) ** 2 + ((yy / max(h - 1, 1) - cy) / 0.9) ** 2
    cone = np.exp(-beam * d)[..., None].astype(np.float32)
    light = ambient * AMB_TINT + (1.0 - ambient) * cone * HEAD_TINT
    unit = reflect * light

    # Auto-exposure & noise
    lit = np.clip(unit * _exposure_for_target(unit, target), 0, 1)
    lit = rng.poisson(lit * peak).astype(np.float32) / peak
    lit = lit + rng.normal(0, read_std, lit.shape).astype(np.float32)
    enc = np.clip(lit, 0, 1) ** (1 / GAMMA)

    # Desaturation
    out = (enc * 255.0).astype(np.uint8)
    gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)[..., None].astype(np.float32)
    return np.clip((1 - desat) * out + desat * gray, 0, 255).astype(np.uint8)

def sample_params(rng):
    name = str(rng.choice(["dusk", "night"]))
    p = dict(PRESETS[name])
    p["target_mean"] *= rng.uniform(0.8, 1.25)
    p["peak"] *= rng.uniform(0.7, 1.4)
    p["read_std"] *= rng.uniform(0.7, 1.6)
    p["beam"] *= rng.uniform(0.8, 1.3)
    p["cx"] = float(np.clip(0.5 + rng.uniform(-0.15, 0.15), 0.25, 0.75))
    p["cy"] = float(np.clip(1.0 + rng.uniform(-0.10, 0.10), 0.85, 1.15))
    return p

src_root = Path("/content/Dataset-3_yolo")
dst_root = Path("/content/Dataset-3_synthetic_night")
rng = np.random.default_rng(42)

for split in ["train", "val", "test"]:
    (dst_root / "images" / split).mkdir(parents=True, exist_ok=True)
    (dst_root / "labels" / split).mkdir(parents=True, exist_ok=True)
    images = sorted(list((src_root / "images" / split).glob("*.jpg")))
    print(f"Transforming {split}: {len(images)} images to synthetic night...")
    for p in tqdm(images, desc=split):
        im = cv2.imread(str(p))
        if im is not None:
            night = to_night(im, rng, **sample_params(rng))
            cv2.imwrite(str(dst_root / "images" / split / p.name), night)
        lbl = src_root / "labels" / split / f"{p.stem}.txt"
        if lbl.exists():
            shutil.copy(str(lbl), str(dst_root / "labels" / split / lbl.name))

yaml_text = f"""path: {dst_root.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['pothole']
"""
(dst_root / "data.yaml").write_text(yaml_text)
print("\nSynthetic Night Dataset Created at /content/Dataset-3_synthetic_night!")


In [ ]:
# Cell 7: Visual Comparison (Day vs. Synthetic Night with Bounding Boxes)
import matplotlib.pyplot as plt

def draw_boxes(img, label_path):
    h, w = img.shape[:2]
    canvas = img.copy()
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                vals = line.strip().split()
                if len(vals) == 5:
                    _, xc, yc, bw, bh = map(float, vals)
                    x1 = int((xc - bw / 2) * w)
                    y1 = int((yc - bh / 2) * h)
                    x2 = int((xc + bw / 2) * w)
                    y2 = int((yc + bh / 2) * h)
                    cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return canvas

sample_imgs = sorted(list((src_root / "images" / "test").glob("*.jpg")))[:3]
fig, axes = plt.subplots(len(sample_imgs), 2, figsize=(10, 4 * len(sample_imgs)))

for i, img_p in enumerate(sample_imgs):
    day_im = cv2.imread(str(img_p))
    night_im = cv2.imread(str(dst_root / "images" / "test" / img_p.name))
    lbl_p = str(src_root / "labels" / "test" / f"{img_p.stem}.txt")

    day_boxed = draw_boxes(day_im, lbl_p)
    night_boxed = draw_boxes(night_im, lbl_p)

    axes[i, 0].imshow(cv2.cvtColor(day_boxed, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Day Original: {img_p.name}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cv2.cvtColor(night_boxed, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"Synthetic Night: {img_p.name}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# SECTION C: DOMAIN TRANSFER TEST (Day-Trained Model on Synthetic Night Test Set)
# ==============================================================================
# Test how much the Day model drops when tested in the dark
best_day_model = YOLO(f"{model.trainer.save_dir}/weights/best.pt")

print("=" * 60)
print("EVALUATING DAY MODEL ON SYNTHETIC NIGHT TEST SET (Domain Gap):")
print("=" * 60)
transfer_res = best_day_model.val(data="/content/Dataset-3_synthetic_night/data.yaml", split="test")

print(f"Day Model on Day Test:            mAP@0.5 = {test_res.box.map50:.4f}")
print(f"Day Model on Synthetic Night Test: mAP@0.5 = {transfer_res.box.map50:.4f}")
drop = test_res.box.map50 - transfer_res.box.map50
print(f"Performance Drop in the dark:     -{drop:.4f} (-{drop*100:.1f}%)")


In [ ]:
# ==============================================================================
# SECTION D: TRAIN FRESH MODEL ON SYNTHETIC NIGHT DATASET
# ==============================================================================
night_model = YOLO("yolov8s.pt")

night_train_results = night_model.train(
    data="/content/Dataset-3_synthetic_night/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="runs/colab_dataset3",
    name="exp_yolov8s_synthetic_night",
    seed=42,
    plots=True
)


In [ ]:
# Cell 10: Evaluate Synthetic-Night Trained Model
best_night_weights = f"{night_model.trainer.save_dir}/weights/best.pt"
best_night_model = YOLO(best_night_weights)

print("=" * 60)
print("EVALUATION OF SYNTHETIC-NIGHT MODEL ON SYNTHETIC-NIGHT TEST SET:")
print("=" * 60)
night_eval_res = best_night_model.val(data="/content/Dataset-3_synthetic_night/data.yaml", split="test")
print(f"Precision:   {night_eval_res.box.p.mean():.4f}")
print(f"Recall:      {night_eval_res.box.r.mean():.4f}")
print(f"mAP@0.5:     {night_eval_res.box.map50:.4f}")
print(f"mAP@0.5:0.95:{night_eval_res.box.map:.4f}")


In [ ]:
# Cell 11: Final Comparison Summary Table
import pandas as pd

comparison_df = pd.DataFrame([
    {"Training Data": "Daytime (Dataset-3)", "Test Data": "Daytime Test", "mAP@0.5": f"{test_res.box.map50:.4f}", "mAP@0.5:0.95": f"{test_res.box.map:.4f}"},
    {"Training Data": "Daytime (Dataset-3)", "Test Data": "Synthetic Night Test", "mAP@0.5": f"{transfer_res.box.map50:.4f}", "mAP@0.5:0.95": f"{transfer_res.box.map:.4f}"},
    {"Training Data": "Synthetic Night", "Test Data": "Synthetic Night Test", "mAP@0.5": f"{night_eval_res.box.map50:.4f}", "mAP@0.5:0.95": f"{night_eval_res.box.map:.4f}"},
])

print("\n" + "=" * 65)
print("             DATASET-3 EXPERIMENT SUMMARY")
print("=" * 65)
print(comparison_df.to_string(index=False))
print("=" * 65 + "\n")


In [ ]:
# Cell 12: Download Complete Experiment Results (Day + Night)
import shutil
from google.colab import files

shutil.make_archive("dataset3_complete_results", "zip", "runs/colab_dataset3")
print("Downloading all training runs, curves, and weights...")
files.download("dataset3_complete_results.zip")
